In [2]:
import os
import json
import pickle
import argparse
import pandas as pd
from tqdm import tqdm
from typing import Any, Dict, List

def load_json(path: str) -> Any:
    with open(path, "r") as f:
        return json.load(f)

def load_pickle(path: str) -> Any:
    with open(path, "rb") as f:
        return pickle.load(f)

def find_camera_name(filename: str) -> str:
    """Heuristic to extract camera name like 'CAM_FRONT' from a filename."""
    parts = filename.split('__')
    if len(parts) > 1:
        return parts[1]
    return "UNKNOWN"

def serialize_array(arr: list) -> str:
    """Convert a list or nested list (e.g., for matrices) into a semicolon-separated string."""
    return ';'.join(map(str, sum(arr, []) if isinstance(arr[0], list) else arr))

def main(args):
    print("Loading original nuScenes data files...")
    sample_data = load_json(args.sample_data_json)
    infos_blob = load_pickle(args.infos_pkl)
    infos = infos_blob["infos"]
    print("Files loaded. Building a lookup map for calibration data...")

    # Create a fast lookup map from sample_data_token -> cam_info
    # This avoids repeatedly searching the large 'infos' list.
    token_to_cam_info = {}
    for sample_info in tqdm(infos, desc="Indexing calibration info"):
        for cam_name, cam_details in sample_info["cams"].items():
            sdt = cam_details.get("sample_data_token")
            if sdt:
                token_to_cam_info[sdt] = cam_details

    print(f"Map created with {len(token_to_cam_info)} entries.")
    
    # Process all samples and collect data
    records = []
    for entry in tqdm(sample_data, desc="Processing samples"):
        sample_token = entry.get("sample_token")
        sample_data_token = entry.get("token")
        filename = entry.get("filename", "")

        if not all([sample_token, sample_data_token, filename]):
            continue

        camera_name = find_camera_name(filename)
        cam_info = token_to_cam_info.get(sample_data_token)

        if cam_info:
            record = {
                "sample_token": sample_token,
                "camera_name": camera_name,
                "sample_data_token": sample_data_token,
                "cam_intrinsic": serialize_array(cam_info["cam_intrinsic"]),
                "sensor2ego_rotation": serialize_array(cam_info["sensor2ego_rotation"]),
                "sensor2ego_translation": serialize_array(cam_info["sensor2ego_translation"]),
            }
            records.append(record)

    # Convert to a DataFrame and save as CSV
    df = pd.DataFrame(records)
    df.to_csv(args.output_csv, index=False)
    print(f"Successfully created metadata file with {len(df)} records.")
    print(f"Saved to: {args.output_csv}")
    print("\nFirst 5 rows of the new file:")
    print(df.head())

if __name__ == "__main__":
    p = argparse.ArgumentParser(description="Create a consolidated metadata CSV from nuScenes data.")
    p.add_argument("--sample_data_json", type=str, required=True, help="Path to sample_data.json")
    p.add_argument("--infos_pkl", type=str, required=True, help="Path to nuscenes_infos_temporal_val.pkl")
    p.add_argument("--output_csv", type=str, default="nuscenes_metadata.csv", help="Path for the output CSV file.")
    
    # Example usage:
    # python create_metadata_file.py \
    #   --sample_data_json /path/to/v1.0-trainval/sample_data.json \
    #   --infos_pkl /path/to/nuscenes_infos_temporal_val.pkl
    
    # Using your hardcoded paths for demonstration if you run without args
    try:
        args = p.parse_args()
    except SystemExit:
        print("Running with default hardcoded paths for demonstration.")
        from types import SimpleNamespace
        args = SimpleNamespace(
             sample_data_json="/home/draiman/Desktop/datasets/nuscenes/v1.0-trainval/sample_data.json",
             infos_pkl="/home/draiman/Desktop/datasets/nuscenes/nuscenes_infos_temporal_val.pkl",
             output_csv="nuscenes_metadata.csv"
        )
    main(args)


usage: ipykernel_launcher.py [-h] --sample_data_json SAMPLE_DATA_JSON
                             --infos_pkl INFOS_PKL [--output_csv OUTPUT_CSV]
ipykernel_launcher.py: error: the following arguments are required: --sample_data_json, --infos_pkl


Running with default hardcoded paths for demonstration.
Loading original nuScenes data files...
Files loaded. Building a lookup map for calibration data...


Indexing calibration info: 100%|███████████████████████████████████████████████████████████████████████████| 1982/1982 [00:00<00:00, 249380.85it/s]


Map created with 11892 entries.


Processing samples: 100%|████████████████████████████████████████████████████████████████████████████| 2631083/2631083 [00:03<00:00, 684455.82it/s]


Successfully created metadata file with 11892 records.
Saved to: nuscenes_metadata.csv

First 5 rows of the new file:
                       sample_token camera_name  \
0  163b70e627854893b88575caf85a56ea   CAM_FRONT   
1  f56a544064a548a39a81f18cc8f633c5   CAM_FRONT   
2  b10f0cd792b64d16a1a5e8349b20504c   CAM_FRONT   
3  cc50000970584ba5aaf14760b1ced696   CAM_FRONT   
4  f286cdaef36648cba3b6a24d2178dc8a   CAM_FRONT   

                  sample_data_token  \
0  d475bfdda0dc4994ad22265771494d5d   
1  be7cd73c9175438484c96f8eb72ea586   
2  6db5cabec0054eab9583bf86c7040f67   
3  2094d1da203e44ce96201dcdb9c9f31f   
4  8ebc55d9dc8b41f68a1c4accb42f8685   

                                       cam_intrinsic  \
0  [1266.41720305    0.          816.26701974];[ ...   
1  [1266.41720305    0.          816.26701974];[ ...   
2  [1266.41720305    0.          816.26701974];[ ...   
3  [1266.41720305    0.          816.26701974];[ ...   
4  [1266.41720305    0.          816.26701974];[ ...   

   

In [3]:
#!/usr/bin/env python3
"""
nuScenes CAM->BEV projection (organized and refactored rewrite)
- Reads pre-processed metadata from a CSV file for fast data lookups.
- Computes ground-plane intersections and splats features into a BEV grid.
Author: you :)
"""
from __future__ import annotations
import os
import argparse
import pandas as pd
from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import numpy as np
import torch
from pyquaternion import Quaternion

# (The data models NuScenesPaths, CameraCalib, and BEVSpec remain the same) @1
@dataclass
class NuScenesPaths:
    metadata_csv_path: str
    camera_feat_dir: str

@dataclass
class CameraCalib:
    K: np.ndarray      # (3, 3)
    R: np.ndarray      # (3, 3) sensor->ego rotation
    T: np.ndarray      # (3,)   sensor->ego translation

@dataclass
class BEVSpec:
    x_min: float = 0.0
    x_max: float = 50.0
    cell_x: float = 0.4
    y_min: float = -25.0
    y_max: float = 25.0
    cell_y: float = 0.25

    @property
    def H(self) -> int:
        return int(np.ceil((self.x_max - self.x_min) / self.cell_x))

    @property
    def W(self) -> int:
        return int(np.ceil((self.y_max - self.y_min) / self.cell_y))

# ------------------------------- I/O and Data Parsing ------------------------- #
def torch_load(path: str) -> Any:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Missing file: {path}")
    return torch.load(path, map_location="cpu")

def extract_calibration_from_row(row: pd.Series) -> CameraCalib: # @1 @2
    """Build intrinsics/extrinsics from a pandas Series (a row from the CSV)."""
    K = np.array(row["cam_intrinsic"].split(';'), dtype=np.float64).reshape(3, 3)
    
    # Rotation is stored as a quaternion [w, x, y, z]
    quat_elements = [float(x) for x in row["sensor2ego_rotation"].split(';')]
    R = Quaternion(quat_elements).rotation_matrix
    
    T = np.array(row["sensor2ego_translation"].split(';'), dtype=np.float64)
    return CameraCalib(K=K, R=R, T=T)

# (All geometry functions like pixel_centers, backproject_to_ground, # @2
# xy_to_bev_indices, and splat_features_to_bev remain exactly the same) # @3
# [Functions from original script would be pasted here]
def pixel_centers(Hf: int, Wf: int, stride: int) -> Tuple[torch.Tensor, torch.Tensor]: # @2
    j = torch.arange(Wf, dtype=torch.float32) + 0.5
    i = torch.arange(Hf, dtype=torch.float32) + 0.5
    u = j * stride
    v = i * stride
    U, V = torch.meshgrid(u, v, indexing="xy")
    U = U.t().contiguous()
    V = V.t().contiguous()
    return U, V

def backproject_to_ground(U: torch.Tensor, V: torch.Tensor, calib: CameraCalib) -> Tuple[np.ndarray, np.ndarray, np.ndarray]: # @2
    Hf, Wf = U.shape
    uv1 = np.stack([U.reshape(-1).numpy(), V.reshape(-1).numpy(), np.ones(Hf * Wf, dtype=np.float32)], axis=0)
    K_inv = np.linalg.inv(calib.K)
    rays_cam = K_inv @ uv1
    rays_cam[1:3, :] *= -1
    rays_ego = calib.R @ rays_cam
    dir_z = rays_ego[2, :]
    eps = 1e-6
    valid_dir = np.abs(dir_z) > eps
    cam_z = calib.T[2]
    s = np.zeros_like(dir_z, dtype=np.float64)
    s[valid_dir] = -cam_z / dir_z[valid_dir]
    mask_valid = valid_dir & (s > 0.0)
    Xg = calib.T[0] + s * rays_ego[0, :]
    Yg = calib.T[1] + s * rays_ego[1, :]
    return Xg, Yg, mask_valid

def xy_to_bev_indices(Xg: np.ndarray, Yg: np.ndarray, valid: np.ndarray, spec: BEVSpec) -> Tuple[np.ndarray, np.ndarray, np.ndarray]: # @2 @3
    ix = ((Xg - spec.x_min) / spec.cell_x).astype(np.int64)
    iy = ((Yg - spec.y_min) / spec.cell_y).astype(np.int64)
    in_bounds = ( (ix >= 0) & (ix < spec.H) & (iy >= 0) & (iy < spec.W) )
    keep = valid & in_bounds
    return ix, iy, keep

def splat_features_to_bev(cam_feat: torch.Tensor, ix: np.ndarray, iy: np.ndarray, keep: np.ndarray, spec: BEVSpec) -> torch.Tensor: # @3
    B, C, Hf, Wf = cam_feat.shape
    N = Hf * Wf
    bev_grid = torch.zeros((B, C, spec.H, spec.W), dtype=cam_feat.dtype)
    if keep.sum() == 0:
        return bev_grid
    feat_flat = cam_feat.reshape(B, C, N)
    ix_v = torch.from_numpy(ix[keep])
    iy_v = torch.from_numpy(iy[keep])
    lin_v = (ix_v * spec.W + iy_v).long()
    bev_flat = bev_grid.view(B, C, spec.H * spec.W)
    mask_idx = torch.from_numpy(np.nonzero(keep)[0]).long()
    src = feat_flat.index_select(dim=2, index=mask_idx)
    idx = lin_v.unsqueeze(0).expand(C, -1)
    for b in range(B):
        bev_flat[b].scatter_add_(dim=1, index=idx, src=src[b])
    return bev_grid

# ---------------------------------- Main ------------------------------------ #
def run_pipeline(
    paths: NuScenesPaths,
    metadata_df: pd.DataFrame,
    target_sample_token: str,
    camera_name: str = "CAM_FRONT",
    fpn_level: int = 0,
    stride_p2: int = 4,
    bev_spec: Optional[BEVSpec] = None,
) -> Dict[str, Any]:
    """Full end-to-end pipeline using pre-processed metadata."""
    bev_spec = bev_spec or BEVSpec()

    # --- Fast lookup from the DataFrame ---
    query = f"sample_token == '{target_sample_token}' and camera_name == '{camera_name}'"
    results = metadata_df.query(query)
    if results.empty:
        raise KeyError(f"No entry found for sample_token='{target_sample_token}' and camera='{camera_name}'")
    
    # Use the first row found
    sample_info = results.iloc[0]

    # --- Calibration ---
    calib = extract_calibration_from_row(sample_info)

    # --- Feature map ---
    feature_filename = f"{target_sample_token}_lvl{fpn_level}.pt"
    feature_path = os.path.join(paths.camera_feat_dir, feature_filename)
    feat_blob = torch_load(feature_path)
    feat = feat_blob["feat"].unsqueeze(0) if feat_blob["feat"].ndim == 3 else feat_blob["feat"]
    B, C, Hf, Wf = feat.shape

    # --- Pixel centers & backproject ---
    U, V = pixel_centers(Hf, Wf, stride=stride_p2) # @2
    Xg, Yg, valid = backproject_to_ground(U, V, calib) # @2

    # --- Discretize to BEV & splat ---
    ix, iy, keep = xy_to_bev_indices(Xg, Yg, valid, bev_spec) # @2 @3
    bev = splat_features_to_bev(feat, ix, iy, keep, bev_spec) # @3

    # --- Debug prints (optional) ---
    print(f"Found sample info for {camera_name} via direct lookup.") # @3 @4
    print("Camera Intrinsic (K):\n", calib.K) # @3 @4
    print("Camera->ego rotation (R):\n", calib.R) # @3 @4
    print("Camera->ego translation (T):\n", calib.T) # @3 @4
    print(f"Input feat shape: {tuple(feat.shape)}") # @4
    print(f"BEV grid: {bev_spec.H} x {bev_spec.W}") # @4
    print("Valid projected pixels:", int(keep.sum())) # @4
    print("BEV tensor shape:", tuple(bev.shape)) # @4

    return {"bev": bev, "calib": calib}

def main():
    # (The argparse and main execution block would be similar, but point to the new CSV) @4 @5
    p = argparse.ArgumentParser(description="nuScenes CAM->BEV projection (Refactored)")
    p.add_argument("--metadata_csv", type=str, default="nuscenes_metadata.csv")
    p.add_argument("--camera_feat_dir", type=str, default="/home/draiman/Desktop/Datasets_nuscenes/validation/fpn/CAM_FRONT")
    p.add_argument("--sample_token", type=str, default="fd8420396768425eabec9bdddf7e64b6")
    p.add_argument("--camera_name", type=str, default="CAM_FRONT")
    p.add_argument("--fpn_level", type=int, default=0)
    p.add_argument("--stride", type=int, default=4)
    # BEV spec args...
    args = p.parse_args()

    paths = NuScenesPaths(
        metadata_csv_path=args.metadata_csv,
        camera_feat_dir=args.camera_feat_dir,
    )
    bev_spec = BEVSpec() # Simplified for brevity

    # Load metadata ONCE
    metadata_df = pd.read_csv(paths.metadata_csv_path)
    print(f"Loaded metadata with {len(metadata_df)} records.")

    run_pipeline(
        paths=paths,
        metadata_df=metadata_df,
        target_sample_token=args.sample_token,
        camera_name=args.camera_name,
        fpn_level=args.fpn_level,
        stride_p2=args.stride,
        bev_spec=bev_spec,
    )

if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] [--metadata_csv METADATA_CSV]
                             [--camera_feat_dir CAMERA_FEAT_DIR]
                             [--sample_token SAMPLE_TOKEN]
                             [--camera_name CAMERA_NAME]
                             [--fpn_level FPN_LEVEL] [--stride STRIDE]
ipykernel_launcher.py: error: unrecognized arguments: -f /home/draiman/.local/share/jupyter/runtime/kernel-9460585c-364b-4a63-90e9-aea2258d6f14.json


SystemExit: 2

/home/draiman/Desktop/Nima/LAVIS_/LAVIS/lavis-env/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
